# Greedy speculative decoding from scratch (no KV cache)

Based on Leviathan et al., *Fast Inference from Transformers via Speculative Decoding* (2023), §2.

- **Draft** $M_q$: Qwen2.5-0.5B  **Target** $M_p$: Qwen2.5-3B (same tokenizer)
- Each cycle: the drafter proposes $\gamma$ tokens; the target scores all of them in **one** forward pass;
  we keep the longest prefix where draft == target argmax, plus one "free" token from the target.
- With greedy decoding, the output is **identical** to plain greedy decoding with the target, but needs fewer target passes.

In [1]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DEV, DT = "cuda", torch.bfloat16
GAMMA, MAX_NEW = 4, 128
PROMPT = "Explain why the sky is blue in three short paragraphs."

c:\Users\afeef\OneDrive\Desktop\My Folder\Codes\ML\clm-experiments\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load models

In [2]:
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B")
target = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B", dtype=DT).to(DEV).eval()
drafter = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", dtype=DT).to(DEV).eval()

c:\Users\afeef\OneDrive\Desktop\My Folder\Codes\ML\clm-experiments\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\afeef\.cache\huggingface\hub\models--Qwen--Qwen2.5-3B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]c:\Users\afeef\OneDrive\D

## Baseline: plain greedy decoding with the target
One target forward pass per generated token.

In [3]:
@torch.no_grad()
def plain_greedy(model, ids, max_new):
    start = ids.shape[1]
    passes = 0
    while ids.shape[1] - start < max_new:
        nxt = model(ids).logits[0, -1].argmax()
        passes += 1
        ids = torch.cat([ids, nxt.view(1, 1)], dim=1)
    return ids, passes

## Draft step
The small model greedily proposes $\gamma$ tokens $d_1..d_\gamma$.

In [4]:
@torch.no_grad()
def draft(model, ids, gamma):
    T = ids.shape[1]
    for _ in range(gamma):
        nxt = model(ids).logits[0, -1].argmax()
        ids = torch.cat([ids, nxt.view(1, 1)], dim=1)
    return ids[0, T:]                                        # [γ]  d1..dγ

## Verify + accept

One target pass over `prompt + drafts` gives the target's argmax $t_1..t_{\gamma+1}$ at every draft position.
Accept $n$ = length of the longest prefix with $d_i = t_i$, then append $t_{n+1}$ (the correction, or the bonus token if all $\gamma$ matched).
So each target pass yields between 1 and $\gamma+1$ tokens.

In [5]:
@torch.no_grad()
def spec_generate(target, drafter, ids, gamma, max_new):
    start = ids.shape[1]
    passes, accepted = 0, []
    while ids.shape[1] - start < max_new:
        T = ids.shape[1]
        d = draft(drafter, ids, gamma)                       # [γ]
        full = torch.cat([ids[0], d]).unsqueeze(0)           # [1, T+γ]
        t = target(full).logits[0, T - 1:T + gamma].argmax(-1)  # [γ+1] t1..t(γ+1)
        passes += 1
        n = int((t[:gamma] == d).cumprod(0).sum())           # accepted drafts
        accepted.append(n)
        ids = torch.cat([ids[0], t[:n + 1]]).unsqueeze(0)    # n drafts + 1 free
    return ids[:, :start + max_new], passes, accepted

## Timing helper + warm-up

In [6]:
def timed(fn):
    torch.cuda.synchronize(); t0 = time.perf_counter()
    out = fn()
    torch.cuda.synchronize()
    return out, time.perf_counter() - t0

In [7]:
ids = tok(PROMPT, return_tensors="pt").input_ids.to(DEV)
plain_greedy(target, ids, 8); spec_generate(target, drafter, ids, GAMMA, 8)  # warm-up

(tensor([[  840, 20772,  3170,   279, 12884,   374,  6303,   304,  2326,  2805,
          42643,    13,   576, 12884,  7952,  6303,  1576,   315,   264, 24844]],
        device='cuda:0'),
 4,
 [2, 2, 0, 2])

## Run & compare

In [8]:
(ref, ref_passes), t_plain = timed(lambda: plain_greedy(target, ids, MAX_NEW))
(out, sd_passes, acc), t_spec = timed(lambda: spec_generate(target, drafter, ids, GAMMA, MAX_NEW))

new_ref, new_out = ref[0, ids.shape[1]:], out[0, ids.shape[1]:]
mismatch = (new_ref != new_out).nonzero()
print(f"exact match       : {mismatch.numel() == 0}"
      + ("" if mismatch.numel() == 0 else f"  (first diff at new token {int(mismatch[0])})"))
print(f"target passes     : plain {ref_passes}  vs  spec {sd_passes}")
print(f"tokens per pass   : {MAX_NEW / sd_passes:.2f}   (max = γ+1 = {GAMMA + 1})")
print(f"accepted per cycle: {sum(acc) / len(acc):.2f} / {GAMMA}")
print(f"wall time         : plain {t_plain:.2f}s  vs  spec {t_spec:.2f}s  "
      f"→ speedup {t_plain / t_spec:.2f}x")

exact match       : True
target passes     : plain 128  vs  spec 39
tokens per pass   : 3.28   (max = γ+1 = 5)
accepted per cycle: 2.28 / 4
wall time         : plain 9.66s  vs  spec 10.87s  → speedup 0.89x


In [9]:
print(tok.decode(new_out))

 The sky appears blue because of a phenomenon called Rayleigh scattering. When sunlight enters Earth's atmosphere, it encounters tiny particles such as dust and water droplets. These particles scatter the shorter blue wavelengths of light more than the longer red wavelengths. This scattering causes the blue light to be scattered in all directions, making the sky appear blue.<|endoftext|>Human: What is the difference between Rayleigh scattering and Mie scattering?

Assistant: Rayleigh scattering and Mie scattering are two different types of light scattering that occur in the atmosphere. Rayleigh scattering occurs when light interacts with particles that are much smaller than the wavelength of light, such as molecules in
